# 01 - Data and knowledge base

**Questions this notebook answers:** what data does the assistant use, why, what can it therefore answer, and how is it cleaned?

```
01 Data + knowledge base  ->  02 Build index  ->  03 RAG pipeline  ->  04 Evaluation
        (you are here)
```

A RAG system can only be as good as the documents it retrieves from, so the knowledge base is chosen to match the questions the assistant is meant to answer, not to be as large as possible.
The notebooks are shipped without outputs. Run them top to bottom to see the numbers.

## Scope

The assistant is an **educational health-information assistant**. It is not an AI doctor.

| | |
|---|---|
| **In scope** | Diseases and conditions: what they are, symptoms, causes and risk factors, prevention, tests and diagnosis, and how they are treated in general |
| **Out of scope** | Specific medicines (doses, side effects, interactions, whether to start, stop or combine), diagnosing the user, personal advice, interpreting someone's test results, and live or current information |

Two levels only: a question either falls inside what the sources can support, or it does not. There is no in-between "partly covered" category, because those are the hardest to test for.

## Sources

| Source | What it is | Why it is in |
|---|---|---|
| **MedQuAD** (`lavita/MedQuAD`) | Question/answer pairs written by NIH websites (NIDDK, CDC, NHLBI, NINDS, Cancer.gov, ...) | Written for the public and organised by question type (symptoms, causes, treatment, prevention, tests...). This copy keeps each document's URL and publisher, which the citations need. |
| **MedlinePlus** (NLM health-topic XML) | One plain-language summary per health topic | Official, with stable URLs that are easy to cite. |

Two things were considered and left out: **PubMedQA** (research abstracts with auto-generated questions; its artificial split alone has 211,269 rows, about twelve times the two sources above combined) and **MedRedQA** (informal Reddit replies about one person's situation). Both fit consumer questions poorly. They were left out on domain fit; their effect on retrieval was not measured.

## Why medicines are out of scope

Medicines are not part of this assistant's scope. This is a design choice, not a limitation of the current data.

* The sources used here describe how conditions are treated in general, not specific drugs. In `lavita/MedQuAD`, the answers of the drug, A.D.A.M., herbs-and-supplements and GARD subsets were removed for copyright (31,034 rows), and no drug-specific question type remains.
* MedlinePlus drug pages are AHFS Consumer Medication Information, copyrighted by ASHP, so they are not ingested.
* FDA drug labels (DailyMed / openFDA) are open, but they are written for clinicians and would need their own ingestion step. They are not part of this project.

So medicines are out of scope. The assistant will abstain on questions about named drugs, and notebook 04 checks that this behaviour holds. Section 5 below confirms the knowledge base indeed does not cover them.

In [ ]:
import io
import re
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from datasets import load_dataset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.kb import (
    BOILERPLATE, build_kb, clean_text, parse_medlineplus_xml, prepare_medlineplus, prepare_medquad,
)

RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
config.KB_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. MedQuAD

Rows without answer text are dropped later. First, how many rows does each NIH publisher have, and how many have an answer?

In [ ]:
# Download MedQuAD from HuggingFace and cache it locally
dataset = load_dataset("lavita/MedQuAD")
raw_medquad = dataset[list(dataset.keys())[0]].to_pandas()
raw_medquad.to_csv(RAW_DIR / "medquad_raw.csv", index=False)

# Count rows per NIH publisher, and how many actually have an answer
raw_medquad["has_answer"] = raw_medquad["answer"].fillna("").str.strip() != ""
by_publisher = raw_medquad.groupby("document_source").agg(rows=("has_answer", "size"), with_answer=("has_answer", "sum"))
print(by_publisher.sort_values("rows", ascending=False).to_string())
print(f"\ntotal: {len(raw_medquad):,} rows, {raw_medquad['has_answer'].sum():,} with an answer")

The publishers with no answers are the ones removed for copyright (including the drug subset). The question types that do have answers:

In [ ]:
# Only rows with an answer are usable; break them down by question type
raw_medquad.loc[raw_medquad["has_answer"], "question_type"].value_counts().to_frame("answered rows")

## 2. MedlinePlus

MedlinePlus publishes one zipped XML file with every health topic. The file name contains the date, so it is read from the download page.

In [ ]:
# The zip URL changes with the date, so scrape it from the download page
page = requests.get("https://medlineplus.gov/xml.html", timeout=30).text
zip_url = re.findall(r"https://medlineplus\.gov/xml/mplus_topics_compressed_\d{4}-\d{2}-\d{2}\.zip", page)[0]
print(zip_url)

# Download the zip, extract the XML inside it, parse it
archive = zipfile.ZipFile(io.BytesIO(requests.get(zip_url, timeout=120).content))
xml_name = [name for name in archive.namelist() if name.endswith(".xml")][0]

raw_medline = parse_medlineplus_xml(archive.read(xml_name))
raw_medline.to_csv(RAW_DIR / "medlineplus_raw.csv", index=False)
print(f"English health topics: {len(raw_medline):,}")

## 3. Clean

Each step is in `src/kb.py` and has a reason:

* **Drop rows without an answer**, broken questions such as `What is (are) ?` (empty topic name), and duplicates (the longer answer is kept).
* **Remove HTML tags** with a pattern that matches real tags only, so clinical text like `<5 mg` survives. MedlinePlus summaries are real HTML, so they go through BeautifulSoup.
* **Cut NIH site text** (`Top of Page`, `Related Pages`, ...) from the end of MedQuAD answers. It is navigation, not content, and would otherwise be retrieved and quoted.
* **Keep provenance.** Every document keeps a source name, a title and a URL. These become the citations users see.

In [ ]:
# Clean each source separately; both helpers live in src/kb.py
medquad = prepare_medquad(raw_medquad)
medline = prepare_medlineplus(raw_medline)
print(f"MedQuAD:     {raw_medquad['has_answer'].sum():>6,} with an answer -> {len(medquad):>6,} documents")
print(f"MedlinePlus: {len(raw_medline):>6,} topics -> {len(medline):>6,} documents")

# Show one example before / after, to see what the cleaning actually removed
pattern = "|".join(re.escape(marker) for marker in BOILERPLATE)
raw_questions = raw_medquad["question"].apply(clean_text)
survived = raw_medquad["answer"].fillna("").str.contains(pattern) & raw_questions.isin(medquad["question"])
if survived.any():
    example = raw_medquad[survived].iloc[0]
    cleaned = medquad.loc[medquad["question"] == clean_text(example["question"]), "text"].iloc[0]
    print("\nraw answer, last 200 characters:\n ...", str(example["answer"])[-200:])
    print("\ncleaned answer, last 200 characters:\n ...", cleaned[-200:])

## 4. The knowledge base

One table, the same columns whatever the source: `doc_id, source, title, url, question, text`.
Answer length decides how many chunks a document becomes in notebook 02.

In [ ]:
# Combine the two sources into one table
kb = build_kb(medquad, medline)
kb["words"] = kb["text"].str.split().str.len()

print(f"{len(kb):,} documents")
print(kb.groupby(kb["source"].str.split(" ").str[0]).agg(documents=("doc_id", "size"), median_words=("words", "median")).to_string())
print(f"documents without a URL: {(kb['url'] == '').sum():,}")

# Quick look at document length distribution
kb["words"].clip(upper=1500).plot.hist(bins=60, figsize=(7, 3), title="Words per document (clipped at 1500)")
plt.tight_layout()
plt.show()
kb.head(3)

## 5. Confirm medicines are not in the knowledge base

The "Why medicines are out of scope" section above makes a claim about the data. This section checks it against the knowledge base we just built.

The evaluation set has 20 `medication` questions, most of them about one named drug ("What are the side effects of metformin?").
For each drug name below, this counts documents that are *about* the drug (name in the title) and documents that only *mention* it.
A document that only mentions a drug while describing a condition's treatment cannot answer questions about its side effects or precautions.

If the count of "documents about it" is zero or close to zero, medicines are indeed out of scope, and the assistant should abstain on these questions.

In [ ]:
eval_questions = pd.read_csv(config.EVAL_PATH)
print(eval_questions.loc[eval_questions["type"] == "medication", "question"].head(8).to_string(index=False))

# For each drug, count docs where it appears in the title vs. anywhere in the text
drugs = ["metformin", "ibuprofen", "paracetamol", "acetaminophen", "amoxicillin", "omeprazole", "levothyroxine",
         "warfarin", "aspirin", "statin", "ACE inhibitor", "antihistamine", "corticosteroid", "insulin"]
titles, texts = kb["title"].str.lower(), kb["text"].str.lower()
pd.DataFrame({
    "documents about it (in title)": [int(titles.str.contains(d.lower(), regex=False).sum()) for d in drugs],
    "documents that mention it": [int(texts.str.contains(d.lower(), regex=False).sum()) for d in drugs],
}, index=drugs)

Read the table like this: many mentions but no documents about the drug means the knowledge base knows that a condition exists and that it is treated, but nothing about a specific medicine's side effects, doses or interactions.

That is why medicines are out of scope. Everything else in the scope (conditions, symptoms, causes, prevention, tests, treatment overviews) is what both sources are made of, and notebook 04 checks it with the evaluation questions.

## 6. Checks and save

In [ ]:
# Sanity checks before saving
assert kb["doc_id"].is_unique
assert (kb["text"].str.len() > 0).all() and (kb["title"].str.len() > 0).all()

kb.drop(columns="words").to_csv(config.KB_PATH, index=False)
print(f"saved {len(kb):,} documents -> {config.KB_PATH.relative_to(ROOT)}")

**Next:** `02_build_index.ipynb` splits these documents into chunks, embeds them and builds the search index.